# AMMS 302 — Week 12: Finding Insights from Big Data (Part 1)
**Google BigQuery (cloud) · DuckDB (offline) · PySynthea synthetic patients**

> เปิดคู่กับ [สไลด์ wk12](./wk12.html) — Lab: สร้างผู้ป่วยสังเคราะห์ด้วย PySynthea → query analytics ด้วย DuckDB/pandas

### 🎯 Learning objectives (CLO2/CLO4)
- อธิบาย BigQuery architecture (columnar, serverless) vs local DW
- สร้าง cohort สังเคราะห์ด้วย PySynthea (tietai-synthea) ได้
- Query CSV/Parquet in-place ด้วย DuckDB แบบออฟไลน์ได้

### 📚 Official references
- **BigQuery:** [docs](https://cloud.google.com/bigquery/docs) · [free sandbox](https://cloud.google.com/bigquery/docs/sandbox) · [loading data](https://cloud.google.com/bigquery/docs/loading-data) · [console](https://console.cloud.google.com/bigquery)
- **PySynthea:** [GitHub TIET-AI/tietai-synthea](https://github.com/TIET-AI/tietai-synthea) · [PyPI](https://pypi.org/project/tietai-synthea/) · [paper](https://tiet.ai/pdf/pysynthea-paper.pdf)
- **DuckDB:** [docs](https://duckdb.org/docs/) · [Python guide](https://duckdb.org/docs/stable/clients/python/overview.html)
- Install: PowerShell `uv add tietai-synthea duckdb` (scoop/uv setup ดู wk05 slide 13)

---


In [ ]:
# Setup — ตรวจ packages (ถ้าไม่มี: uv add tietai-synthea duckdb แล้ว restart kernel)
import importlib, sys
print("python", sys.version.split()[0])
for pkg in ['synthea','duckdb','pandas']:
    try:
        m = importlib.import_module(pkg if pkg!='tietai-synthea' else 'synthea')
        print(f"✅ {pkg}", getattr(m,'__version__','?'))
    except ModuleNotFoundError:
        print(f"❌ {pkg} — run in PowerShell: uv add {pkg}")

## §1 PySynthea — สร้าง synthetic cohort (สไลด์ 05)
Python-native reimplementation ของ Synthea™ (MITRE) — ไม่ต้อง Java · export FHIR R4/CSV · deterministic seed


In [ ]:
# §1 generate 50 patients (seed=42 → reproducible)
try:
    from synthea import Generator, GeneratorOptions
    opts = GeneratorOptions()
    opts.population_size = 50
    opts.seed = 42
    gen = Generator(opts)
    gen.run()
    print(gen.stats)
except ModuleNotFoundError:
    print("⚠️ install first: uv add tietai-synthea")
except Exception as e:
    print("API note:", e, "— see https://github.com/TIET-AI/tietai-synthea#quick-start")
    print("ทางเลือก CLI: uv run synthea -p 50 --state California")

## §2 Export & load — FHIR bundles / CSV (สไลด์ 05–06)
PySynthea ให้ pandas-friendly exports — pattern: materialize → DataFrame → analyze ทันที


In [ ]:
# §2 fallback demo dataset (ถ้า generator ใช้ไม่ได้ในเครื่อง): สังเคราะห์ mini-cohort ด้วย faker-style logic เอง
import numpy as np, pandas as pd
rng = np.random.default_rng(42)
n = 2000
cohort = pd.DataFrame({
  "patient_id": np.arange(1, n+1),
  "gender":     rng.choice(['M','F'], n),
  "age":        rng.integers(18, 90, n),
  "hba1c":      rng.normal(6.8, 1.5, n).round(1).clip(3.5, 15),
  "sbp":        rng.normal(132, 20, n).round(0),
  "dm_dx":      False,
})
cohort['dm_dx'] = cohort['hba1c'] >= 6.5
cohort.to_csv("synthetic_cohort.csv", index=False)
print(cohort.shape); cohort.head()

## §3 DuckDB — SQL บนไฟล์โดยตรง (สไลด์ 06)
`read_csv_auto` → query ไฟล์ใหญ่โดยไม่ import RAM ทั้งก้อน — syntax SQL มาตรฐาน


In [ ]:
import duckdb
con_d = duckdb.connect()  # in-memory

# §3.1 direct-on-file query
q1 = """
 SELECT gender,
        COUNT(*) n,
        ROUND(AVG(hba1c),2) avg_a1c,
        ROUND(100.0*SUM(dm_dx)/COUNT(*),1) pct_dm
 FROM read_csv_auto('synthetic_cohort.csv')
 GROUP BY gender ORDER BY gender"""
display(con_d.execute(q1).df())

# §3.2 age bands + CASE
q2 = """
 SELECT CASE WHEN age<40 THEN '18-39' WHEN age<60 THEN '40-59' ELSE '60+' END band,
        COUNT(*) n, SUM(dm_dx::INT) dm_n
 FROM read_csv_auto('synthetic_cohort.csv')
 GROUP BY band ORDER BY band"""
display(con_d.execute(q2).df())

## §4 Insights: prevalence & comorbidity patterns (สไลด์ 08)


In [ ]:
# §4 prevalence by decade + uncontrolled share
q3 = """
 SELECT age/10*10 AS decade, COUNT(*) n,
        ROUND(AVG(sbp),0) avg_sbp,
        ROUND(100.0*SUM(CASE WHEN hba1c>7 THEN 1 ELSE 0 END)/COUNT(*),1) pct_uncontrolled
 FROM read_csv_auto('synthetic_cohort.csv')
 GROUP BY decade ORDER BY decade"""
prev = con_d.execute(q3).df()
display(prev)

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6,3))
ax.bar(prev['decade'].astype(str), prev['pct_uncontrolled'], color='teal')
ax.set_title('% HbA1c > 7 (uncontrolled DM proxy) by decade'); ax.set_xlabel('decade'); ax.set_ylabel('%')
plt.tight_layout(); plt.show()
# 💡 ค่า trend ขึ้นตามอายุ = insight แบบ epidemiology จริง (จาก synthetic!)

## §5 BigQuery path (สไลด์ 03–04) — เมื่อพร้อมใช้ cloud
1) เปิด [sandbox ฟรี](https://cloud.google.com/bigquery/docs/sandbox) (ไม่ต้องใส่บัตร)  
2) Console → create dataset `amms302`  
3) Upload `synthetic_cohort.csv` → auto schema  
4) Run SQL เดียวกัน (แทน read_csv_auto ด้วย table name)  
5) Dry-run ดู bytes ก่อนรันจริง (ประหยัด quota)

> ⚠️ ตาม wk10: อย่า upload raw patient data — synthetic only!

```sql
-- BigQuery version of q1:
SELECT gender, COUNT(*) n, AVG(hba1c) avg_a1c
FROM `your-project.amms302.synthetic_cohort`
GROUP BY gender;
```


In [ ]:
# Cleanup
con_d.close(); print("duckdb closed ✅ — files kept: synthetic_cohort.csv")

### ✅ Self-check
- generator.stats แสดง total_generated=50 (หรือ fallback 2000 rows)
- DuckDB group-by-gender ได้ 2 แถว + pct_dm
- decade chart แสดง %uncontrolled ขึ้นตามอายุ
- อธิบายได้ว่าทำไมใช้ synthetic แทน raw (wk10 tie-in)

### 📝 Homework 12
generate cohort 500 คน (seed ตัวเลขรหัสนักศึกษา) → ตาราง top-3 decade ที่ dm prevalence สูงสุด + กราฟ — ส่ง .ipynb + csv

---
### 🔗 Specs
[BigQuery](https://cloud.google.com/bigquery/docs) · [sandbox](https://cloud.google.com/bigquery/docs/sandbox) · [PySynthea](https://github.com/TIET-AI/tietai-synthea) · [DuckDB python](https://duckdb.org/docs/stable/clients/python/overview.html)
